# <center>Procesamiento V2 para Redes Neuronales</center>


Este notebook arma una variante orientada a modelos densos, en especial redes neuronales. Mantiene el mismo contrato metodologico del resto, pero agrega una imputacion jet-aware aprendida solo sobre train y un escalado robusto sobre las features numericas.


## Imports y utilidades


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_try_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        direct_candidate = candidate / "higgs_preprocessing_v2.py"
        nested_candidate = candidate / "10.try" / "higgs_preprocessing_v2.py"
        if direct_candidate.exists():
            return candidate
        if nested_candidate.exists():
            return candidate / "10.try"
    raise FileNotFoundError("No se pudo ubicar 10.try/higgs_preprocessing_v2.py")


try_dir = find_try_dir()
if str(try_dir) not in sys.path:
    sys.path.insert(0, str(try_dir))

from higgs_preprocessing_v2 import (
    VARIANT_DEFINITIONS,
    get_higgs_data_dir,
    run_preprocessing_variant,
    save_preprocessing_artifacts,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


## Variante elegida
- No descarta columnas por missing.
- Imputa numericas usando medianas aprendidas por grupo de `PRI_jet_num`, con reglas fisicas para columnas que estructuralmente no existen sin jets.
- Agrega flags `is_missing_*`.
- Escala numericas con `RobustScaler` y hace one-hot de `PRI_jet_num`.


In [2]:
variant_key = "nn_dense"
variant = VARIANT_DEFINITIONS[variant_key]
artifacts = run_preprocessing_variant(variant_key=variant_key)
output_dir = get_higgs_data_dir() / "processed" / variant.output_dir_name

print(f"Variante: {variant.title}")
print(f"Directorio de salida esperado: {output_dir}")
print("Descripcion:")
print(variant.description)


Variante: Procesamiento V2 - Pipeline para Redes Neuronales
Directorio de salida esperado: /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_nn_dense
Descripcion:
Agrega indicadores de missing, aplica imputacion jet-aware aprendida solo en train, codifica PRI_jet_num y escala features numericas con RobustScaler.


## Resumen del procesamiento


In [3]:
metadata = artifacts["metadata"]

print("Resumen del split temprano:")
display(artifacts["split_summary_df"].round(4))

print("Missingness en train crudo luego del reemplazo -999 -> NaN:")
display(artifacts["train_missing_summary_df"].head(15))

print("Metadata principal del procesamiento:")
for key, value in metadata.items():
    print(f"- {key}: {value}")

print("Primeras filas del train procesado:")
display(artifacts["X_train_processed"].head())
print("Etiquetas de train:")
display(artifacts["y_train"].head().to_frame(name="Label"))
print("Columnas con flags de missing:")
display(pd.DataFrame({"feature": artifacts["missing_indicator_source_columns"]}).head(20))


Resumen del split temprano:


,split,rows,class_b,class_s,weight_sum
0,train,225000,147900.0,77100.0,370822.7398
1,valid,25000,16433.0,8567.0,40869.0961
2,test_external,550000,NaN,NaN,NaN


Missingness en train crudo luego del reemplazo -999 -> NaN:


,feature,missing_count,missing_pct
0,DER_deltaeta_jet_jet,159718,70.99
1,DER_lep_eta_centrality,159718,70.99
2,DER_mass_jet_jet,159718,70.99
3,DER_prodeta_jet_jet,159718,70.99
4,PRI_jet_subleading_eta,159718,70.99
5,PRI_jet_subleading_phi,159718,70.99
6,PRI_jet_subleading_pt,159718,70.99
7,PRI_jet_leading_eta,89987,39.99
8,PRI_jet_leading_phi,89987,39.99
9,PRI_jet_leading_pt,89987,39.99


Metadata principal del procesamiento:
- variant_key: nn_dense
- variant_title: Procesamiento V2 - Pipeline para Redes Neuronales
- description: Agrega indicadores de missing, aplica imputacion jet-aware aprendida solo en train, codifica PRI_jet_num y escala features numericas con RobustScaler.
- valid_size: 0.1
- random_state: 42
- raw_feature_count: 30
- processed_feature_count: 44
- processed_train_nan_count: 0
- processed_valid_nan_count: 0
- processed_test_nan_count: 0
- expects_missing_after_processing: False
Primeras filas del train procesado:


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,DER_pt_ratio_lep_tau,DER_met_phi_centrality,DER_lep_eta_centrality,PRI_tau_pt,PRI_tau_eta,PRI_tau_phi,PRI_lep_pt,PRI_lep_eta,PRI_lep_phi,PRI_met,PRI_met_phi,PRI_met_sumet,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,is_missing_DER_mass_MMC,is_missing_DER_deltaeta_jet_jet,is_missing_DER_mass_jet_jet,is_missing_DER_prodeta_jet_jet,is_missing_DER_lep_eta_centrality,is_missing_PRI_jet_leading_pt,is_missing_PRI_jet_leading_eta,is_missing_PRI_jet_leading_phi,is_missing_PRI_jet_subleading_pt,is_missing_PRI_jet_subleading_eta,is_missing_PRI_jet_subleading_phi,PRI_jet_num_0,PRI_jet_num_1,PRI_jet_num_2,PRI_jet_num_3
0,-0.023990,-0.071820,0.116793,-0.008610,0.000000,0.000000,0.000000,0.210687,0.589046,-0.139629,-0.424242,-0.372242,0.000000,0.606917,0.124741,0.438850,-0.249841,0.104202,-0.160484,-0.300599,-0.619812,-0.084723,-0.306535,0.000000,0.000000,0.00000,0.000000,0.000000,-0.205067,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
1,0.000000,0.296772,0.439683,-0.270591,0.000000,0.000000,0.000000,0.064122,-0.250087,-0.113398,-0.134680,-0.374733,0.000000,0.334126,0.251552,-0.421606,0.164672,0.595400,0.341996,0.027891,-0.046974,-0.227613,-0.306535,0.000000,0.000000,0.00000,0.000000,0.000000,-0.205067,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
2,-0.062532,-0.463134,-0.237248,0.393952,0.000000,0.000000,0.000000,-0.265140,0.301229,0.230077,-0.156004,0.538790,0.000000,-0.006178,0.614839,0.486224,-0.207950,0.331630,-0.439000,0.343582,-0.452419,0.457947,0.584714,-0.713292,0.163958,0.00000,0.000000,0.000000,0.368479,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
3,0.011691,0.407822,0.257697,0.062245,0.000000,0.000000,0.000000,0.188295,-0.246617,0.027942,-0.291807,-0.372954,0.000000,0.341889,0.232338,-0.346482,-0.130445,0.560761,0.148185,0.398595,0.602695,-0.101734,0.059570,0.475796,0.064771,0.00000,0.000000,0.000000,0.030533,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
4,-0.276728,0.003561,-0.206995,0.611735,1.255102,1.794034,-1.890792,-0.446819,0.026161,0.727446,0.763187,0.618505,0.378449,-0.205361,-0.186225,-0.360951,0.495497,-0.438387,-0.123983,0.305970,-0.268307,0.186427,0.931474,-0.365873,0.528442,0.94937,1.258947,-0.244039,0.869289,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


Etiquetas de train:


,Label
0,s
1,b
2,s
3,b
4,s


Columnas con flags de missing:


,feature
0,DER_mass_MMC
1,DER_deltaeta_jet_jet
2,DER_mass_jet_jet
3,DER_prodeta_jet_jet
4,DER_lep_eta_centrality
5,PRI_jet_leading_pt
6,PRI_jet_leading_eta
7,PRI_jet_leading_phi
8,PRI_jet_subleading_pt
9,PRI_jet_subleading_eta


## Guardado de artefactos
Se persisten los `csv` procesados del split train/valid/test, los IDs, los pesos, la metadata del split y el `joblib` con el pipeline ajustado.


In [4]:
saved_paths = save_preprocessing_artifacts(artifacts, output_dir=output_dir)

print(f"Directorio de salida: {output_dir}")
print(f"Cantidad de artefactos guardados: {len(saved_paths)}")
for path in saved_paths:
    print(f"- {path}")


Directorio de salida: /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_nn_dense
Cantidad de artefactos guardados: 19
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_nn_dense/X_train_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_nn_dense/X_valid_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_nn_dense/X_test_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_nn_dense/y_train.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_nn_dense/y_valid.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_nn_dense/train_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_nn_dense/valid_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_nn_dense/train_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_nn_dense/valid_event_id.csv
- /home/santig14/fing/